# 05 — Advanced Patterns

This notebook brings together everything from the previous notebooks
into more realistic scenarios:

- ML-style pipelines with preprocessing and models
- Optimizing pipeline hyperparameters with Study
- Working with 2D data (matrices)
- Multi-step processing chains

## 5.1 — An ML-style pipeline

Let's build a realistic preprocessing + model pipeline that:
1. Normalizes features (learns mean/std)
2. Applies a simple linear model (learns weights from training data)

In [ ]:
from soma import Filter, Pipeline, Study, search

class FeatureScaler(Filter):
    """Normalizes each feature column independently."""

    def fit(self, x, y=None):
        # x is a 2D list: [[f1, f2, ...], [f1, f2, ...], ...]
        n_features = len(x[0])
        means = [sum(row[j] for row in x) / len(x) for j in range(n_features)]
        stds = [
            (sum((row[j] - means[j]) ** 2 for row in x) / len(x)) ** 0.5
            for j in range(n_features)
        ]
        return {"means": means, "stds": stds}

    def forward(self, x, state):
        means = state["means"]
        stds = state["stds"]
        return [
            [(row[j] - means[j]) / max(stds[j], 1e-8) for j in range(len(row))]
            for row in x
        ]

class LinearModel(Filter):
    """Simple weighted sum model with regularization parameter."""
    reg: float = search(0.001, 10.0, scale="log")

    def __init__(self, reg=1.0):
        super().__init__(reg=reg)

    def fit(self, x, y=None):
        # Simplified: learn average weights from data
        n_features = len(x[0]) if x else 1
        # In reality you'd solve a linear system. Here we simulate.
        weights = [1.0 / n_features] * n_features
        return {"weights": weights, "bias": 0.0}

    def forward(self, x, state):
        weights = state["weights"]
        bias = state["bias"]
        return [sum(row[j] * weights[j] for j in range(len(row))) + bias for row in x]

# Build and train the pipeline
pipeline = Pipeline([
    ("scaler", FeatureScaler()),
    ("model", LinearModel(reg=0.1)),
])

# Training data: 5 samples, 3 features
train_x = [
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
    [10.0, 11.0, 12.0],
    [13.0, 14.0, 15.0],
]

pipeline.fit(train_x)
print(f"Pipeline fitted: {pipeline.filter_names()}")

# Predict on new data
test_x = [[2.0, 3.0, 4.0], [8.0, 9.0, 10.0]]
predictions = pipeline.predict(test_x)
print(f"Predictions: {[f'{p:.4f}' for p in predictions]}")

## 5.2 — Optimizing a pipeline with Study

The real power: combine `search()` descriptors on filters with `Study`
to automatically find the best hyperparameters for your pipeline.

In [ ]:
class Preprocessor(Filter):
    """Preprocessing with searchable clip threshold."""
    clip_max: float = search(1.0, 100.0, scale="log")

    def __init__(self, clip_max=50.0):
        super().__init__(clip_max=clip_max)

    def fit(self, x, y=None):
        return {}

    def forward(self, x, state):
        return [min(v, self.clip_max) for v in x]

class Predictor(Filter):
    """Model with searchable learning rate and regularization."""
    lr: float = search(0.001, 1.0, scale="log")
    alpha: float = search(0.0, 1.0)

    def __init__(self, lr=0.01, alpha=0.5):
        super().__init__(lr=lr, alpha=alpha)

    def fit(self, x, y=None):
        return {"baseline": sum(x) / max(len(x), 1)}

    def forward(self, x, state):
        # Simulate: prediction quality depends on lr and alpha
        return [v * self.lr + state["baseline"] * self.alpha for v in x]

# Combine search spaces from both filters
combined_space = Preprocessor._soma_search_space + Predictor._soma_search_space
print("Combined search space:")
for dim in combined_space:
    print(f"  {dim['name']}: {dim['type']} [{dim.get('low', '')}, {dim.get('high', '')}]")

# Define the objective: build a pipeline with the given params, evaluate it
train_data = [10.0, 20.0, 30.0, 40.0, 50.0]
test_data = [15.0, 25.0, 35.0]
target = [14.0, 24.0, 34.0]  # "ground truth"

def pipeline_objective(params):
    # Build pipeline with trial parameters
    pipe = Pipeline([
        Preprocessor(clip_max=params["clip_max"]),
        Predictor(lr=params["lr"], alpha=params["alpha"]),
    ])
    pipe.fit(train_data)
    preds = pipe.predict(test_data)

    # Compute MSE
    mse = sum((p - t) ** 2 for p, t in zip(preds, target)) / len(target)
    return {"mse": mse}

# Run Bayesian optimization
study = Study(
    name="pipeline_tuning",
    search_space=combined_space,
    strategy="bayesian",
    n_trials=30,
    objectives=[("mse", "minimize")],
    seed=42,
)

study.run(pipeline_objective)

best = study.best_trial
print(f"\nBest trial ({study.n_trials} evaluated):")
print(f"  clip_max = {best['params']['clip_max']:.3f}")
print(f"  lr       = {best['params']['lr']:.6f}")
print(f"  alpha    = {best['params']['alpha']:.3f}")
print(f"  MSE      = {best['metrics']['mse']:.4f}")

## 5.3 — Reusable filter library pattern

A good practice is to build a library of composable filters, each doing
one thing well. Then compose them into pipelines for different tasks.

In [ ]:
# --- Filter library ---

class ClampFilter(Filter):
    """Clamps values to [low, high] range."""
    _kind = "stateless"

    def __init__(self, low=0.0, high=1.0):
        super().__init__(low=low, high=high)

    def forward(self, x, state):
        return [max(self.low, min(self.high, v)) for v in x]

class OffsetFilter(Filter):
    """Learns the mean, subtracts it (centering)."""

    def fit(self, x, y=None):
        return {"offset": sum(x) / len(x)}

    def forward(self, x, state):
        return [v - state["offset"] for v in x]

class ScaleFilter(Filter):
    """Scales by a fixed factor."""
    _kind = "stateless"

    def __init__(self, factor=1.0):
        super().__init__(factor=factor)

    def forward(self, x, state):
        return [v * self.factor for v in x]

class ThresholdFilter(Filter):
    """Converts to binary: 1 if above threshold, 0 otherwise."""
    _kind = "stateless"

    def __init__(self, threshold=0.0):
        super().__init__(threshold=threshold)

    def forward(self, x, state):
        return [1.0 if v > self.threshold else 0.0 for v in x]

# --- Compose into different pipelines ---

# Pipeline A: center → scale → clamp (feature preprocessing)
preprocess = Pipeline([
    OffsetFilter(),
    ScaleFilter(factor=0.1),
    ClampFilter(low=-1.0, high=1.0),
])
preprocess.fit([10.0, 20.0, 30.0, 40.0, 50.0])
print("Preprocess:", preprocess.predict([15.0, 25.0, 45.0, 100.0]))

# Pipeline B: center → threshold (binary classification)
classify = Pipeline([
    OffsetFilter(),
    ThresholdFilter(threshold=0.0),
])
classify.fit([10.0, 20.0, 30.0, 40.0, 50.0])
print("Classify:  ", classify.predict([15.0, 25.0, 35.0, 45.0]))
# Below mean → 0, above mean → 1

## 5.4 — JSON data pipelines

Soma isn't limited to numeric data. Filters can process JSON/dict data
for text processing, feature extraction, or API-style transformations.

In [ ]:
class TextTokenizer(Filter):
    """Splits text into tokens and builds a vocabulary."""
    _kind = "trainable"

    def fit(self, x, y=None):
        # x is a dict with a "texts" field
        texts = x["texts"]
        vocab = {}
        for text in texts:
            for word in text.lower().split():
                if word not in vocab:
                    vocab[word] = len(vocab)
        return {"vocab": vocab, "vocab_size": len(vocab)}

    def forward(self, x, state):
        vocab = state["vocab"]
        texts = x["texts"]
        tokenized = []
        for text in texts:
            tokens = [vocab.get(w.lower(), -1) for w in text.split()]
            tokenized.append(tokens)
        return {"tokens": tokenized, "vocab_size": state["vocab_size"]}

class TokenCounter(Filter):
    """Counts tokens per document."""
    _kind = "stateless"

    def forward(self, x, state):
        tokens = x["tokens"]
        return {"counts": [len(t) for t in tokens], "total": sum(len(t) for t in tokens)}

# JSON pipeline: tokenize → count
nlp_pipe = Pipeline([
    ("tokenizer", TextTokenizer()),
    ("counter", TokenCounter()),
])

corpus = {"texts": ["hello world", "soma is a runtime", "hello soma"]}
nlp_pipe.fit(corpus)
print("Trained on corpus")

result = nlp_pipe.predict({"texts": ["hello soma world", "new words here"]})
print(f"Token counts: {result['counts']}")
print(f"Total tokens: {result['total']}")

## 5.5 — What's next: the Rust runtime

Everything in these notebooks runs through the Rust runtime via PyO3 bindings.
Under the hood, Soma provides much more:

| Feature | Description | Crate |
|---|---|---|
| **Graph compilation** | Pipelines compile to ExecutionPlans with parallelism detection | `soma-compiler` |
| **Parallel execution** | Independent branches run on separate threads | `soma-runtime` |
| **Tiered caching** | Memory → Disk → Remote, with automatic promotion | `soma-runtime` |
| **Distribution** | Plans can be scheduled across workers | `soma-compiler` |
| **Stream execution** | Chunk-based processing with state checkpointing | `soma-runtime` |
| **MCP server** | 13-tool MCP interface for code agents | `soma-mcp` |
| **Remote workers** | Isolated Python environments per pipeline | `soma-worker` |

```
Python (you are here)
  ↓ PyO3
soma-runtime (Pipeline, Cache, Study, Stream)
  ↓
soma-compiler (Graph → ExecutionPlan)
  ↓
soma-core (Filter, Value, CacheKey, Schema — the contracts)
```